Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools
!pip install imbalanced-learn

Calling the Libraries:

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
import glob
import os
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter

Train

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (60, 128)
PATCH_SIZE = 10
stride = 10
FINGER_NUMS = [1, 2, 3, 4, 5, 6]
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]
NUM_COMPONENTS = 47  # Number of 2DPCA components

TRAIN_INDICES = {
    1: [1, 2, 3],
    2: [1, 2, 3],
    3: [1, 2, 3],
    4: [1, 2]
}

def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        if transitions <= 2:
            table[i] = sum(min_rotation)
        else:
            table[i] = P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === FEATURE EXTRACTION (Fused LBP) ===
train_lbp_features = []
train_labels = []

print("\n🔄 Extracting fused RIU2-LBP features from UTFVP...")

subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for img_num, aug_list in TRAIN_INDICES.items():
        for aug_id in aug_list:
            fused_vector = []
            suffix = f"{img_num}_{aug_id}_Augmented.png"
            print(f"\n➡️ Subject {subj} - Image {suffix}")

            for finger in FINGER_NUMS:
                fname = f"{subj}_{finger}_{suffix}"
                img_path = os.path.join(subject_path, fname)
                print(f"  📥 Loading: {img_path}")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

                if img is None:
                    print(f"  ❌ Missing image: {img_path}")
                    continue

                img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img).astype(np.float64) / 255.0
                img = (img - np.mean(img)) / (np.std(img) + 1e-8)

                for y in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                    for x in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        block_hist = []
                        for R, P in LBP_CONFIGS:
                            hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            block_hist.extend(hist)
                        fused_vector.extend(block_hist)

            if len(fused_vector) > 0:
                label = f"{subj}_fused_img{img_num}_aug{aug_id}"
                train_lbp_features.append(fused_vector)
                train_labels.append(label)

# === NORMALIZE AND CONVERT TO ARRAY ===
train_lbp_features = np.array(train_lbp_features, dtype=np.float32)
train_lbp_features = normalize(train_lbp_features, norm='l2')
train_labels = np.array(train_labels)

print("\n✅ RIU2-LBP feature matrix shape:", train_lbp_features.shape)

# === 2DPCA ===
original_dim = train_lbp_features.shape[1]
height = int(np.sqrt(original_dim))
while original_dim % height != 0:
    height -= 1
width = original_dim // height
print(f"\n🔁 Reshaping each feature to: ({height}, {width})")
train_2d = [f.reshape(height, width) for f in train_lbp_features]

def compute_2dpca(images_2d, num_components):
    print("\n⚙️ Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        if i < 3:
            print(f"  ➕ Sample {i+1} contribution added")

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ 2DPCA projection matrix shape: {eig_vecs.shape}")
    return eig_vecs

W = compute_2dpca(train_2d, NUM_COMPONENTS)

projected_features = []
for i, img in enumerate(train_2d):
    feat = img @ W
    projected_features.append(feat)
    if i < 3:
        print(f"🧮 Projected shape of sample {i+1}: {feat.shape}")

train_lbp_pca = np.array([f.flatten() for f in projected_features])
train_lbp_pca = normalize(train_lbp_pca, norm='l2')

print("\n✅ Final 2DPCA-transformed shape:", train_lbp_pca.shape)
print("✅ Number of principal components used:", W.shape[1])


Test:

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (60, 128)
PATCH_SIZE = 10
stride = 10
FINGER_NUMS = [1, 2, 3, 4, 5, 6]
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]

# ✅ Protocol 1 Test Set (Strategy 1: Fused)
TEST_FILES = [
    ("1", ""), ("2", ""), ("3", ""), ("4", ""), ("4", "3_Augmented")
]

# === RIU2 MAPPING ===
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        table[i] = sum(min_rotation) if transitions <= 2 else P + 1
    return table

mapping_dict = {p: get_riu2_mapping(p) for _, p in LBP_CONFIGS}

def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === FUSED FEATURE EXTRACTION ===
test_lbp_features = []
test_labels = []

print("\n🧪 Extracting fused RIU2-LBP test features from UTFVP (Strategy 1 – Protocol 1)...")

subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for img_num, aug in TEST_FILES:
        fused_vector = []
        if aug:
            suffix = f"{img_num}_{aug}.png"   # ✅ FIXED: Correct augmented filename
            label_suffix = f"img{img_num}_{aug}"
        else:
            suffix = f"{img_num}.png"
            label_suffix = f"img{img_num}_orig"

        print(f"\n➡️ Subject {subj} - Image {label_suffix}")

        for finger in FINGER_NUMS:
            fname = f"{subj}_{finger}_{suffix}"
            img_path = os.path.join(subject_path, fname)
            print(f"  📥 Loading: {img_path}")

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"  ❌ Missing image: {img_path}")
                continue

            img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
            img = cv2.fastNlMeansDenoising(img, h=10)
            img = cv2.equalizeHist(img).astype(np.float64) / 255.0
            img = (img - np.mean(img)) / (np.std(img) + 1e-8)

            for y in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                for x in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                    block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                    block_hist = []
                    for R, P in LBP_CONFIGS:
                        hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                        block_hist.extend(hist)
                    fused_vector.extend(block_hist)

        if len(fused_vector) > 0:
            label = f"{subj}_fused_test_{label_suffix}"
            test_lbp_features.append(fused_vector)
            test_labels.append(label)

# === FINALIZE TEST SET ===
test_lbp_features = np.array(test_lbp_features, dtype=np.float32)
test_lbp_features = normalize(test_lbp_features, norm='l2')
test_labels = np.array(test_labels)

print("\n✅ Test feature extraction complete!")
print("🔢 Feature matrix shape:", test_lbp_features.shape)
print("🟢 Example test labels:", test_labels[:5])

# === PROJECT TEST FEATURES USING 2DPCA MATRIX (W) ===
original_dim = test_lbp_features.shape[1]
height = int(np.sqrt(original_dim))
while original_dim % height != 0:
    height -= 1
width = original_dim // height

print(f"\n📐 Reshaping test features to: ({height}, {width})")
test_2d = [f.reshape(height, width) for f in test_lbp_features]

# ✅ Ensure 'W' is defined from the training phase
try:
    projected_test = []
    for i, img in enumerate(test_2d):
        feat = img @ W
        projected_test.append(feat)
        if i < 3:
            print(f"🧪 Projected test image {i+1}: {feat.shape}")

    test_flat_features = np.array([f.flatten() for f in projected_test])
    test_flat_features = normalize(test_flat_features, norm='l2')

    print("\n✅ Test feature projection complete!")
    print("📐 Final projected shape:", test_flat_features.shape)

except NameError:
    print("\n❌ Error: Projection matrix W is not defined. Load it from training.")


Benchmarking:

In [ ]:

# === CLASSIFICATION ===
correct_matches = 0
total_tests = len(test_flat_features)

print("\n🔍 Starting classification using Manhattan distance (2DPCA)...")

for i in range(total_tests):
    test_vec = test_flat_features[i]
    true_label = test_labels[i]

    distances = np.sum(np.abs(train_lbp_pca - test_vec), axis=1)
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]

    true_id = true_label.split('_')[0]
    pred_id = predicted_label.split('_')[0]

    if pred_id == true_id:
        correct_matches += 1
        match_result = "✅"
    else:
        match_result = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Subject Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests})")


Session Independent R5

In [ ]:
import numpy as np
from collections import defaultdict

# === Configuration ===
ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating Session-Independent CMC (Rank-1 & Rank-5) — Strategy 1: Fused Fingers...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    test_label = test_labels[i]

    # ✅ Extract subject only (ignore session for session-independent evaluation)
    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_id = test_subject  # No session in ID

    # Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_pca - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    matched = False
    for r in range(1, max(ranks) + 1):
        candidate_label = train_labels[sorted_indices[r - 1]]
        parts = candidate_label.split('_')
        candidate_subject = parts[0]
        candidate_id = candidate_subject  # No session in ID

        if candidate_id == test_id and not matched:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            matched = True

# === Final CMC Results
for k in ranks:
    accuracy = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy (Subject only): {accuracy:.2f}%")


Session Independent CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ✅ Helper function: extract subject only (ignore session and finger)
def extract_subject(label):
    return label.split('_')[0]  # subject ID only

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_flat_features)

print("📊 Calculating Session-Independent CMC Curve (Matching by Subject Only)...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_label = test_labels[i]
    true_subject = extract_subject(true_label)

    # Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(train_lbp_pca - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # Find the first correct match
    for r in range(max_rank):
        candidate_label = train_labels[sorted_indices[r]]
        candidate_subject = extract_subject(candidate_label)

        if candidate_subject == true_subject:
            rank_correct[r:] += 1
            break

# ✅ Normalize to percentage
cmc_curve = (rank_correct / total_tests) * 100

# ✅ Plotting the CMC curve
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Independent CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# ✅ Print key rank accuracies
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Independent Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# === Initialize lists
all_scores = []
all_labels = []

# === Pairwise score computation for Session-Independent Verification (Strategy 1: Fused Fingers)
for test_idx in range(len(test_flat_features)):
    test_vec = test_flat_features[test_idx]
    test_label = test_labels[test_idx]
    test_parts = test_label.split('_')
    test_subject = test_parts[0]  # ✅ Use only subject
    test_id = test_subject

    for train_idx in range(len(train_lbp_pca)):
        train_vec = train_lbp_pca[train_idx]
        train_label = train_labels[train_idx]
        train_parts = train_label.split('_')
        train_subject = train_parts[0]  # ✅ Use only subject
        train_id = train_subject

        # Skip self-comparison (optional but recommended)
        if test_label == train_label:
            continue

        # Similarity score: negative Manhattan distance
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # Ground truth: genuine if subject matches (session ignored)
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize similarity scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
scores = (scores - scores.min()) / (scores.max() - scores.min())
# === Toggle SMOTE ===
use_smote = True  # Set True if you want to apply SMO

if use_smote:
    smote = SMOTE(random_state=42)
    scores_2d = scores.reshape(-1, 1)  # Reshape to 2D: (n_samples, 1)
    scores_2d, labels = smote.fit_resample(scores_2d, labels)
    scores = scores_2d.ravel()  # Flatten back to 1D for thresholding
    print("🧪 After SMOTE label distribution:", Counter(labels))
# === Threshold Sweeping to Find Best F1 Score
best_f1 = best_thresh = best_prec = best_rec = 0

for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final Classification at Optimal Threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Print Summary Report
print("🔍 Summary (Session-Independent Verification — Strategy 1: Fused Fingers)")
print("📎 Feature: LBP((8,1),(16,1),(8,2)) + (2D)^2PCA")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
